In [12]:
%%writefile llm_client.py

Overwriting llm_client.py


In [11]:
%%writefile test_llm_client.py

Overwriting test_llm_client.py


In [15]:
!python -m unittest -v test_llm_client

test_both_satisfy_protocol (test_llm_client.Tests.test_both_satisfy_protocol) ... ok
test_factory (test_llm_client.Tests.test_factory) ... ok
test_fallback_local_down_uses_hosted (test_llm_client.Tests.test_fallback_local_down_uses_hosted) ... ok
test_gemini (test_llm_client.Tests.test_gemini) ... ok
test_non_retryable_raises (test_llm_client.Tests.test_non_retryable_raises) ... ok
test_retries_then_succeeds (test_llm_client.Tests.test_retries_then_succeeds) ... ok
test_same_call_both_providers (test_llm_client.Tests.test_same_call_both_providers) ... ok
test_streaming_both (test_llm_client.Tests.test_streaming_both) ... ok

----------------------------------------------------------------------
Ran 8 tests in 2.136s

OK


In [16]:
!ls -l test_llm_client.py
!python -c "import test_llm_client; print(test_llm_client.__file__); print([x for x in dir(test_llm_client.Tests) if x.startswith('test_')])"

-rw-r--r-- 1 root root 7950 Nov  9 07:46 test_llm_client.py
/content/test_llm_client.py
['test_both_satisfy_protocol', 'test_factory', 'test_fallback_local_down_uses_hosted', 'test_gemini', 'test_non_retryable_raises', 'test_retries_then_succeeds', 'test_same_call_both_providers', 'test_streaming_both']


In [17]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

curl: (22) The requested URL returned error: 504


In [18]:
!apt-get update -qq
!apt-get install -y zstd pciutils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
pciutils is already the newest version (1:3.10.0-2build1).
0 upgraded, 0 newly installed, 0 to remove and 83 not upgraded.


In [19]:
!curl -fsSL --retry 5 --retry-delay 5 --retry-all-errors https://ollama.com/install.sh -o install_ollama.sh
!bash install_ollama.sh

curl: (22) The requested URL returned error: 504
curl: (22) The requested URL returned error: 504
curl: (22) The requested URL returned error: 504
curl: (22) The requested URL returned error: 504
curl: (22) The requested URL returned error: 504
curl: (22) The requested URL returned error: 504
bash: install_ollama.sh: No such file or directory


In [20]:
!set -o pipefail
!curl -fL --retry 3 --retry-delay 5 \
https://ollama.com/download/ollama-linux-amd64.tar.zst \
-o ollama-linux-amd64.tar.zst

!tar --zstd -xf ollama-linux-amd64.tar.zst -C /usr

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1472M  100 1472M    0     0  89.3M      0  0:00:16  0:00:16 --:--:--  120M


In [21]:
!ollama --version

In [22]:
import os
import subprocess
import time
import urllib.request

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("ollama.log", "w"),
    stderr=subprocess.STDOUT,
    env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"}
)

for i in range(30):
    try:
        urllib.request.urlopen(
            "http://127.0.0.1:11434/api/tags",
            timeout=2
        )
        print("Ollama server is running!")
        break
    except Exception:
        time.sleep(1)
else:
    print("Ollama server failed to start.")
    print(open("ollama.log").read())

Ollama server is running!


In [25]:
MODEL = "llama3.2"
!ollama pull {MODEL}

In [26]:
!ollama list

NAME               ID              SIZE      MODIFIED       
llama3.2:latest    a80c4f17acd5    2.0 GB    32 seconds ago    


In [27]:
import importlib
import llm_client

importlib.reload(llm_client)

from llm_client import create_client, Message

MODEL = "llama3.2"

local = create_client(
    f"ollama:{MODEL}",
    base_url="http://127.0.0.1:11434"
)

response = local.chat(
    [Message.user("In one sentence, what is Google Colab?")],
    max_tokens=100
)

print("AI Response:", response.text)
print("Provider:", response.provider)
print("Model:", response.model)
print("Usage:", response.usage)

AI Response: Google Colab is a free, cloud-based, and open-source development environment that allows users to write and execute Python code in a web-based interface, with access to GPU acceleration, libraries, and tools for data science and machine learning.
Provider: ollama
Model: llama3.2
Usage: Usage(input_tokens=35, output_tokens=48)


In [28]:
for chunk in local.stream(
    [Message.user("Count from 1 to 10, separated by commas.")],
    max_tokens=60
):
    print(chunk, end="", flush=True)

1, 2, 3, 4, 5, 6, 7, 8, 9, 10

In [29]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

gemini = create_client("gemini:gemini-3.5-flash-lite")

response = gemini.chat(
    [Message.user("In one sentence, what is Artificial Intelligence?")],
    max_tokens=100
)

print("AI Response:", response.text)
print("Provider:", response.provider)
print("Model:", response.model)
print("Usage:", response.usage)

AI Response: Artificial Intelligence is the simulation of human intelligence by computer systems, enabling them to learn, reason, perceive, and perform tasks autonomously.
Provider: gemini
Model: gemini-3.5-flash-lite
Usage: Usage(input_tokens=10, output_tokens=26)


In [30]:
!python -m unittest -v test_llm_client

test_both_satisfy_protocol (test_llm_client.Tests.test_both_satisfy_protocol) ... ok
test_factory (test_llm_client.Tests.test_factory) ... ok
test_fallback_local_down_uses_hosted (test_llm_client.Tests.test_fallback_local_down_uses_hosted) ... ok
test_gemini (test_llm_client.Tests.test_gemini) ... ok
test_non_retryable_raises (test_llm_client.Tests.test_non_retryable_raises) ... ok
test_retries_then_succeeds (test_llm_client.Tests.test_retries_then_succeeds) ... ok
test_same_call_both_providers (test_llm_client.Tests.test_same_call_both_providers) ... ok
test_streaming_both (test_llm_client.Tests.test_streaming_both) ... ok

----------------------------------------------------------------------
Ran 8 tests in 2.173s

OK
